In [5]:
import pandas as pd
import numpy as np
import os

I've tried to design this notebook so that we can 'run all' once the following cell, which gives a list containing all the 'Transee' data.

In [6]:
csv_files = os.listdir('raw_data')
dataframe = [pd.read_csv(f'raw_data/{csv}') for csv in csv_files]

/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_13596/3738552529.py:2: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = [pd.read_csv(f'raw_data/{csv}') for csv in csv_files]
/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_13596/3738552529.py:2: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = [pd.read_csv(f'raw_data/{csv}') for csv in csv_files]
/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_13596/3738552529.py:2: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = [pd.read_csv(f'raw_data/{csv}') for csv in csv_files]
/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_13596/3738552529.py:2: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = [pd.read_csv(f'raw_data/{csv}') for csv in csv_files]


In [7]:
# Drop columns I'm not sure what to do with.
for df in dataframe:
    for remove in ['Unnamed: 0', 'Gap', 'Time', 'Riders after stop']:
        if remove in df.columns:
            df.drop(remove, axis=1, inplace=True)

In [8]:
def find_datetime(day: str | float, time: str | float) -> pd.Timestamp | float:
    if type(day) != str or type(time) != str:
        return np.nan
    if time[0] == ' ':
        time = time[1:]
    return pd.Timestamp(day+' '+time)


In [9]:
for df in dataframe:
    df['Time'] = df.Schedule.str[-10:]
    df['scheduled time'] = df.apply(lambda x: find_datetime(day=x['day'], time=x['Time']), axis=1)
    df.drop(columns=['day', 'Time', 'day of the week'], inplace=True)

We're going to eventually split the data into eastbound vs westbound, so we'll add a column to address this.

In [10]:
def east_or_west(dest: str| float) -> str| float:
    if type(dest) != str:
        return np.nan
    dest = dest.lower()
    if 'east' in dest:
        return 'E'
    else:
        return 'W'


In [11]:
for df in dataframe:
    df['EB/WB'] = df['Destination'].apply(east_or_west)
    df.drop('Destination', axis=1, inplace=True)

In [12]:
def min_delay(schedule: str|float) -> int|float:
    if type(schedule) != str:
        return np.nan

    schedule = schedule.lower()
    if 'ahead' in schedule:
        min_marker = schedule.find(':')
        hour_marker = schedule[:min_marker].rfind(' ')
        if hour_marker == -1:
            hour_marker = 0
        minute = int(schedule[hour_marker:min_marker])
    elif 'behind' in schedule:
        min_marker = schedule.find(':')
        hour_marker = schedule[:min_marker].rfind(' ')
        if hour_marker == -1:
            hour_marker = 0
        minute = -int(schedule[hour_marker:min_marker])
    else:
        minute = 0

    return minute

for df in dataframe:
    df['min delay'] = df['Schedule'].apply(min_delay)
    df.drop('Schedule', axis=1, inplace=True)

Finally, 'cleaned_df' will be all of the dataframes merged together.

In [13]:
cleaned_df = pd.concat(dataframe, ignore_index=True)

In [14]:
# I am going to elect to drop the times where there is no scheduling info for now
cleaned_df.dropna(axis=0, subset=['scheduled time'], inplace=True)

In [15]:
# Finally, we need to time order cleaned_df.
cleaned_df.sort_values(by=['scheduled time'], ignore_index=True, inplace=True)

In [16]:
cleaned_df.head()

,Vehicle,Headway,scheduled time,EB/WB,min delay
0,4418.0,10:00,2025-01-01 00:00:00,W,0.0
1,4552.0,10:00,2025-01-01 00:00:17,E,5.0
2,4400.0,11:15,2025-01-01 00:00:39,W,1.0
3,4591.0,10:00,2025-01-01 00:00:40,E,13.0
4,4481.0,8:20,2025-01-01 00:00:49,W,-11.0


The goal is to use the TTC summary data as the basis of our dataset, with columns appended
from cleaned_df. First we need to add a 'EB/WB' column.

In [17]:
summary_df = pd.read_csv('../Munroe-Streetcar-Info/Streetcar_data_cleaned_up.csv')
remove_columns = ['Unnamed: 0', 'First dep NB or WB', 'First dep SB or EB', 'Last dep NB or WB', 'Last dep SB or EB', 'Route']
for column in remove_columns:
    if column in summary_df.columns:
        summary_df.drop(column, axis=1, inplace=True)

In [18]:
main_df_East = summary_df.copy()
main_df_East['EB/WB'] = ['E' for _ in range(main_df_East.shape[0])]
main_df_West = summary_df.copy()
main_df_West['EB/WB'] = ['W' for _ in range(main_df_West.shape[0])]

main_df = pd.concat([main_df_East, main_df_West], ignore_index=True)
main_df = main_df.sort_values(by=['time period start', 'time period end'], ignore_index=True)
main_df.head()

,date,No. of Veh,Service interval,Run time (min),Term time (min),Avg. spd (km/h),Time of Day (morning: 0-late evening: 4)*,Weekday=0/Sat=1/Sun=2,RT dist (km),Interruption,time period start,time period end,EB/WB
0,2023-01-14,12,10:00,106,14,18.4,0,1,32.56,NaN,2023-01-14 04:00:00,2023-01-14 08:00:00,E
1,2023-01-14,12,10:00,106,14,18.4,0,1,32.56,NaN,2023-01-14 04:00:00,2023-01-14 08:00:00,W
2,2023-01-14,16,09:30,138,14,14.2,1,1,32.56,NaN,2023-01-14 08:00:00,2023-01-14 12:00:00,E
3,2023-01-14,16,09:30,138,14,14.2,1,1,32.56,NaN,2023-01-14 08:00:00,2023-01-14 12:00:00,W
4,2023-01-14,20,08:30,156,14,12.5,2,1,32.56,NaN,2023-01-14 12:00:00,2023-01-14 19:00:00,E


In [19]:
# Let's turn the interruption 'nans' into 0-1s.
def binary_interruption(interruption: str|float) -> int:
    if type(interruption) != str:
        return 0
    else:
        return 1

main_df['Interruption'] = main_df['Interruption'].apply(binary_interruption)

# Turns the time periods into pandas timestamps
main_df['time period start'] = main_df['time period start'].apply(lambda x: pd.Timestamp(x))
main_df['time period end'] = main_df['time period end'].apply(lambda x: pd.Timestamp(x))

In [20]:
# We should filter main_df by the dates coming from cleaned_df.
earliest_timestamp = cleaned_df['scheduled time'].min()
latest_timestamp = cleaned_df['scheduled time'].max()
main_df = main_df[(main_df['time period start'] >= earliest_timestamp) & (main_df['time period end'] <= latest_timestamp)]

In [21]:
def count_bunch(df: pd.DataFrame) -> int:
    current_time = [
        (cleaned_df['scheduled time'].iloc[row] + pd.Timedelta(minutes=cleaned_df['min delay'].iloc[row])).timestamp()
        for row in range(df.shape[0])
    ]
    number_bunch = 0
    for row in range(1,df.shape[0]):
        # This uses the fact that the cleaned_df is time ordered.
        if current_time[row] - current_time[row-1] <= 120:
            number_bunch += 1
    return number_bunch

def count_gap(df: pd.DataFrame) -> int:
    current_time = [
        (cleaned_df['scheduled time'].iloc[row] + pd.Timedelta(minutes=cleaned_df['min delay'].iloc[row])).timestamp()
        for row in range(df.shape[0])
    ]
    number_gap = 0
    for row in range(1,df.shape[0]):
        # This uses the fact that the cleaned_df is time ordered.
        if abs(current_time[row] - current_time[row-1]) >= 19*60:
            number_gap += 1
    return number_gap

In [22]:
total_delay = []
number_bunch = []
number_gap = []

for row in range(main_df.shape[0]):
    time_start = main_df['time period start'].iloc[row]
    time_end = main_df['time period end'].iloc[row]
    E_or_W = main_df['EB/WB'].iloc[row]
    current_data = cleaned_df[(cleaned_df['scheduled time'] >= time_start)
                            & (cleaned_df['scheduled time'] < time_end)
                            & (cleaned_df['EB/WB'] == E_or_W)]

    number_bunch.append(count_bunch(current_data))

    number_gap.append(count_gap(current_data))

# We will not add all the times the streetcar was ahead of schedule
    total_delay.append(sum(current_data['min delay'].apply(lambda x: max(x,0))))

main_df['bunch'] = number_bunch
main_df['total delay'] = total_delay
main_df['gap'] = number_gap

In [23]:
main_df.head()

,date,No. of Veh,Service interval,Run time (min),Term time (min),Avg. spd (km/h),Time of Day (morning: 0-late evening: 4)*,Weekday=0/Sat=1/Sun=2,RT dist (km),Interruption,time period start,time period end,EB/WB,bunch,total delay,gap
7180,2025-01-01,16,10:00,150,10,12.1,0,0,30.13,0,2025-01-01 04:00:00,2025-01-01 09:00:00,E,401,1851.0,13
7181,2025-01-01,16,10:00,150,10,12.1,0,0,30.13,0,2025-01-01 04:00:00,2025-01-01 09:00:00,W,31,159.0,5
7182,2025-01-01,17,10:00,161,9,11.2,1,0,30.13,0,2025-01-01 09:00:00,2025-01-01 15:00:00,E,1318,8165.0,13
7183,2025-01-01,17,10:00,161,9,11.2,1,0,30.13,0,2025-01-01 09:00:00,2025-01-01 15:00:00,W,367,2299.0,13
7184,2025-01-01,19,10:00,180,10,10.0,2,0,30.13,0,2025-01-01 15:00:00,2025-01-01 19:00:00,E,875,4836.0,13


Since our data is big, we split the dataset into time of day/ weekday vs weekend.

In [29]:
# name of the time period column (it's very long)
TIME = [x for x in summary_df.columns if 'Time' in x][0]
# name of the week/sat/sun column (it's also very long)
WEEK = [x for x in summary_df.columns if 'Week' in x][0]

week_translator = {0: 'weekday', 1: 'saturday', 2: 'sunday'}

df_split = dict()
for time in range(5):
    for week in range(3):
        df_split[time, week] = main_df[(main_df[TIME] == time) & (main_df[WEEK] == week)]

Finally, let's write all the data into their own csv files. The final files are not that large because
the rows are split by date/time period.

In [32]:
for key, df in df_split.items():
    df.to_csv(f'processed_data/schedule_data_{week_translator[key[1]]}_{key[0]}.csv', index=False)